# LangSmith Evaluation in Practice

A fluent answer can hide a broken tool, an unsupported assumption, or a missing step. Practical evaluation connects real execution traces to repeatable tests: identify a behavior, measure it, change the agent, and check the same cases again.


## Datasets, Evaluators, and Experiments

Agent outputs can vary even when the input is unchanged. Several outputs may be equally acceptable, and different sequences of tool calls may reach the same result. Exact string equality is useful for tightly specified outputs, but it is usually too restrictive for an open-ended answer. Unit tests can also check properties and invariants; they are not limited to string matching.

:::{div}
Dataset
: A collection of test examples, each containing the input and optionally a reviewed reference output or other expected properties.
:::

:::{div}
Evaluator
: A procedure that scores a run against a defined criterion using its inputs, outputs, intermediate calls, or resulting state.
:::

:::{div}
Experiment
: A batch of runs over a dataset, together with evaluator results and the configuration needed to interpret and compare them.
:::

For a support agent, the input might include an order number and user context. One evaluator checks whether the correct order lookup was called. Another checks whether the answer reports the returned delivery date. These are different behaviors and should produce separate scores.

| Component | Support-agent example | What to preserve |
| --- | --- | --- |
| Dataset example | “Where is order 4821?” | Request, relevant user context, expected status if known |
| Evaluator | Correct order ID passed to lookup | Exact criterion and failure explanation |
| Experiment | Run version A across all examples | Agent version, evaluator version, settings, and per-example results |


## Read the Trace Before Changing the Prompt

A trace exposes the sequence behind a reply. Start with a manageable sample of real runs to learn which requests occur and where failures arise. A small initial dataset can help debug a behavior, but cannot establish a reliable production failure rate on its own.

```{figure} figures/langsmith-run-tree.svg
:alt: A support request branches into a model decision, an order lookup with arguments and results, and a final model reply.
The tool result is an intermediate observation, not the final answer. Inspect both when diagnosing a failure.
```

A useful trace records the input, available context, model calls, tool arguments, tool results, and final output. Timing and token usage help explain speed and resource use. If a tool result is wrong, a polished final answer may simply reproduce that wrong result accurately.

:::{admonition} A Trace Output Is Not Automatically a Reference Answer
:class: warning

When traces become dataset examples, their original outputs may be copied too. Those outputs can contain the very bug being investigated. Review them before treating them as ground truth, or use an evaluator that checks an independent expected property.
:::


## Choose What the Evaluator Observes

| Evaluation target | Example criterion | Suitable evidence |
| --- | --- | --- |
| One step | Lookup used the requested order ID | Tool name and parsed arguments |
| Final response | Reply states the returned delivery date | Answer and trusted tool result |
| Trajectory | Lookup occurred before a factual answer | Ordered calls and messages |
| End state | A requested refund actually exists | Verified backend state |

Code checks suit criteria with explicit rules. A judge model suits criteria requiring interpretation, provided the rubric is clear and its judgments are calibrated. Pairwise comparison is an evaluation format: either humans or a model can compare two outputs. Human review helps establish the intended standard and investigate disagreements.

::::{div}
:class: reading-comparison
:::{div}
### Deterministic Checks

Validate a schema, compare an identifier, check an allowed ingredient set, or inspect call order. These checks are repeatable for fixed recorded inputs, although a live backend may change between runs.
:::
:::{div}
### Rubric-Based Judgments

Assess whether a reply addresses the request, whether instructions are coherent, or whether a response meets an explicitly defined quality standard. Keep separate criteria separate so failures remain interpretable.
:::
::::


## DinnerBot: The Bug Was in the Tool

DinnerBot receives a kitchen inventory, calls `getPantry`, and returns a dish, ingredient list, and instructions. In the demonstration, the kitchen contains pasta, eggs, parmesan, and spinach. The suggested recipe also uses ingredients such as butter, garlic, and milk.

The trace reveals the cause: the tool adds an overly generous built-in staples list. The model's answer is consistent with the tool output, but the tool output incorrectly expands what the user has available. This is a tool-contract failure rather than evidence that the model ignored its context.

```{figure} figures/langsmith-pantry.svg
:alt: The user's four ingredients enter a pantry tool; unsupported staples are added there and then flow into the generated recipe.
The error enters before generation. Fixing only the wording of the final answer leaves the inventory contract broken.
```

:::{admonition} Define the Allowed Set Independently
:class: note

An assumed-staples policy must be explicit. The demonstration uses a reduced staples list, but no ingredient should be universally assumed available for every user. Build the expected set from the application's approved policy or confirmed inventory—not from the faulty tool output being tested.
:::

### A Small Ingredient Evaluator

Let $K$ be the confirmed kitchen ingredients, $A$ the explicitly allowed staples, and $R$ the recipe ingredients. The unsupported set is

$$
U=R\setminus(K\cup A).
$$

The ingredient check passes exactly when $U$ is empty. This is a check of ingredient membership, not a complete assessment of recipe quality.

```python
def ingredient_check(kitchen, allowed_staples, recipe):
    def normalize(name):
        return name.strip().casefold()

    allowed = {normalize(x) for x in kitchen + allowed_staples}
    used = {normalize(x["name"]) for x in recipe["ingredients"]}
    unsupported = sorted(used - allowed)
    return {
        "score": int(not unsupported),
        "unsupported": unsupported,
    }
```

This illustrative evaluator assumes already validated lists and ingredient objects. Real data may require controlled aliases, quantities, units, and checks for ingredients mentioned only in the instructions. A synonym mismatch should not be confused with genuinely unavailable food.

:::{admonition} Worked Ingredient Check
:class: tip ds-example-card

For a test fixture, let the confirmed inventory be pasta, eggs, parmesan, and spinach, with no extra staples allowed. A recipe uses pasta, eggs, parmesan, butter, and garlic.

$$
U=\{\text{butter},\text{garlic}\},\qquad \mathrm{score}=0.
$$

After correcting the tool, a recipe containing only pasta, eggs, parmesan, and spinach passes this check. It still needs separate evaluation of its instructions and suitability.
:::


## Compare the Fix with the Baseline

Use the same examples and evaluator definitions before and after a change. Save the baseline before fixing the agent so improvement is measurable. The DinnerBot demonstration reports that its corrected staples policy removes the ingredient failures in the tested batch; this is a result on those examples, not a general guarantee.

Inspect paired outcomes instead of only an average. An improved overall pass rate can hide previously correct behavior that regressed.

:::{admonition} An Average Can Hide a Regression
:class: tip ds-example-card

In an illustrative set of 20 cases:

| Per-case change | Number of cases |
| --- | ---: |
| Pass → pass | 14 |
| Fail → pass | 4 |
| Pass → fail | 1 |
| Fail → fail | 1 |

The pass rate rises from $15/20=75\%$ to $18/20=90\%$. Nevertheless, one formerly passing case now fails. Inspect it before accepting the change; its importance may outweigh the average improvement.
:::

When runs vary, repeat relevant cases and compare distributions rather than relying on one lucky run. Keep cost and latency alongside quality, and separate evaluator failures from agent failures.


## Replace “Good” with Specific Criteria

The demonstration first asks a judge whether a recipe is “good.” Human reviewers consider specific dimensions, while the broad judge can reward an appealing description without checking the same requirements. The remedy is to state those dimensions explicitly.

```{figure} figures/langsmith-judge-matrix.svg
:alt: One broad good-or-bad rating is separated into two axes, realism and safety, producing four possible combinations.
Independent judgments preserve the reason a recipe fails instead of compressing every concern into one vague score.
```

| Judge | Scope | Boundary |
| --- | --- | --- |
| Realism | Does this describe a recognizable, plausible dinner with coherent steps? | Do not reward formal writing over a casual but clear recipe |
| Safety | Do the instructions contain a safety issue under the defined review rubric? | Do not substitute taste or presentation for the requested assessment |

The session reports agreement on 6 of 8 broad judgments and on all 16 labels after using two narrow criteria for those eight recipes. The denominators differ: 8 overall labels versus 16 criterion-specific labels. This small demonstration illustrates clearer diagnosis, not proof that the judge will generalize perfectly.

:::{admonition} Calibration Needs Fresh Cases
:class: important

Use human-reviewed examples to refine the rubric, then check it on separate examples. Track false passes as well as false failures. An LLM safety judgment is an evaluation signal, not certification that following a recipe is safe.
:::


## Offline and Online Evaluation Form a Loop

Offline evaluation tests a controlled dataset before a change is shipped. Online evaluation scores live traces to discover failures that the offline set did not cover. A production failure can become a reviewed regression case for the next release.

```{figure} figures/langsmith-feedback-loop.svg
:alt: Build and offline testing lead to release; monitoring and failure review return new test cases to the offline dataset.
Monitoring improves the next test set. Review and sanitize traces before promoting them into reusable examples.
```

An offline evaluator can run online if the required evidence is present. The ingredient check can work when inventory and recipe are captured in the trace. An evaluator requiring a reference answer cannot simply assume one exists for a new user request.

| Offline | Online |
| --- | --- |
| Controlled inputs and repeatable fixtures | Real requests and unexpected combinations |
| Optional reviewed references | Usually no prepared reference answer |
| Compare candidate versions | Detect emerging failure patterns |
| Diagnose known regression cases | Supply new cases for offline testing |

Keep the evaluator version with the score. Changing the rubric or judge can change the measured pass rate even when the agent stays the same.


## Evaluate Work, Not Just Completion Claims

For coding or action-taking agents, the result may live outside the response. “I fixed the test” must be checked against the repository and test results. A longer-task benchmark can package instructions, an isolated environment, and a verifier that inspects the resulting state.

A fresh environment per run limits carryover: one run should not inherit another run's modified files or completed actions. The session presents Harbor and Terminal-Bench as examples of this approach; the general evaluation structure matters independently of a particular harness or integration.

:::{admonition} Recipe Remixer as a Consistency Test
:class: tip ds-example-card

The session's follow-up scenario is an agent that substitutes ingredients and rewrites cooking instructions. An illustrative failure is replacing chicken with tofu in the ingredient list while leaving “add the chicken” in the steps.

A useful test checks agreement between the requested constraint, the substitution tool result, the ingredient list, and the rewritten instructions. A correct substitution call alone is insufficient. Test both a successful substitution and a case where no suitable substitution is available; the agent should communicate the limitation rather than invent success.
:::
